## Лабораторный практикум по курсу «Разговорный ИИ и большие языковые модели», Университет ИТМО, 2026  
**Лабораторная работа №2.**

**Цель работы:** Обучить 2 диалоговые модели (decoder-only и encoder-decoder) предсказывать ответ системы на запрос пользователя. 

**Задачи:**

1. Выбрать подходящую модель https://huggingface.co/models.
2. Выбрать подходящий датасет https://huggingface.co/datasets.

Выполнить шаги 3-8 для decoder-only и для encoder-decoder модели.

3. Проверить работу модели в сценарии близком к выбранному датасету в zero-shot режиме.
4. Посчитать метрики bleu и meteor исходной модели на валидационном сабсете выбранного датасета.
5. Дообучить модель на обучающем сабсете выбранного датасета.
6. Проверить работу модели в сценарии близком к выбранному датасету.
7. Посчитать метрики bleu и meteor обученной модели на валидационном сабсете выбранного датасета.
8. Сравнить метрики обученной и исходной моделей.

### 0. Импорты

In [1]:
import os
import nltk
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import List, Dict, Any
from sacrebleu.metrics import BLEU
from datasets import load_dataset, DatasetDict
from nltk.translate.meteor_score import meteor_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, TrainingArguments, Trainer


/home/ext-ivanova-mk@ad.speechpro.com/ITMO/sem_2/aae_venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Выбор моделей

**Decoder-only:** [`ai-forever/rugpt3small_based_on_gpt2`](https://huggingface.co/ai-forever/rugpt3small_based_on_gpt2) — русскоязычная GPT-2 small.

- Архитектура decoder-only с каузальным маскированием
- Генерирует текст авторегрессивно
- *Первичная задача GPT* - продолжить текст

**Encoder-decoder:** [`ai-forever/ruT5-base`](https://huggingface.co/ai-forever/ruT5-base) — русскоязычная T5-base.

- Архитектура encoder-decoder, схема text-to-text
- Эффективна для задач «вход - выход»
- *Первичная задача T5* - восстановить повреждённый текст

### Характеристики

| Параметр    | rugpt3small       | ruT5-base                  |
|-------------|-------------------|----------------------------|
| Архитектура | Decoder-only      | Encoder-decoder            |
| Параметры   | 125 млн           | 222 млн                    |
| Словарь     | 50 257 (BPE)      | 32 128 (SentencePiece)     |

#### 1.1. decoder-only

In [2]:
gpt_tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
gpt_model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

print("GPT params:", sum(p.numel() for p in gpt_model.parameters()))

GPT params: 125231616


#### 1.2. encoder-decoder

In [3]:
t5_tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-base")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("ai-forever/ruT5-base")

print("T5 params:", sum(p.numel() for p in t5_model.parameters()))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5 params: 222903552


### 2. Выбор датасета и подготовка даннных

**Датасет:** [`d0rj/ru-instruct`](https://huggingface.co/datasets/d0rj/ru-instruct) - русскоязычный инструктивный датасет

- Подходит для задачи «запрос - ответ»
- Формат `conversations` с ролями `system` / `user` / `assistant`
- Очищен от артефактов перевода и дублей

**Разбиение (после обрезки и фильтрации):**

| Split | GPT    | T5     |
|-------|--------|--------|
| train | 18 673 | 19 780 |
| val   | 926    | 983    |

скачивание

In [4]:
dataset = load_dataset("d0rj/ru-instruct", split="train")
print(dataset)

Dataset({
    features: ['conversations', 'source'],
    num_rows: 753650
})


Извлечение system/user/assistant


In [5]:
def extract_dialog(example):
    conv = example['conversations']
    system, user, assistant = "", "", ""
    for turn in conv:
        if turn['role'] == 'system':
            system = turn['content']
        elif turn['role'] == 'user':
            user = turn['content']
        elif turn['role'] == 'assistant':
            assistant = turn['content']
    return system, user, assistant

Обрезка датасета: train/val

In [6]:
SEED = 42
TRAIN_SIZE = 20000
VAL_SIZE = 1000

dataset = dataset.shuffle(seed=SEED)
train_raw = dataset.select(range(TRAIN_SIZE))
val_raw = dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))

print(f"Train raw: {len(train_raw)}, Val raw: {len(val_raw)}")

Train raw: 20000, Val raw: 1000


Фильтрация и форматирование для GPT

In [7]:
MAX_LEN_GPT = 512

def format_for_gpt(example):
    system, user, assistant = extract_dialog(example)
    if not user or not assistant:
        return {"input_ids": [], "prompt_len": 0, "valid": False}

    assistant_ids = gpt_tokenizer(assistant, truncation=False, add_special_tokens=False)["input_ids"]
    if len(assistant_ids) > MAX_LEN_GPT:
        return {"input_ids": [], "prompt_len": 0, "valid": False}

    prompt = f"{system}\n\nПользователь: {user}\nАссистент:"
    max_prompt_len = MAX_LEN_GPT - len(assistant_ids)
    prompt_ids = gpt_tokenizer(prompt, truncation=True, max_length=max_prompt_len,
                                add_special_tokens=False)["input_ids"]

    full_ids = prompt_ids + assistant_ids + [gpt_tokenizer.eos_token_id]
    full_ids = full_ids[:MAX_LEN_GPT]

    if len(full_ids) < 10:  # слишком короткий пример
        return {"input_ids": [], "prompt_len": 0, "valid": False}

    return {"input_ids": full_ids, "prompt_len": len(prompt_ids), "valid": True}

Фильтрация и форматирование для T5

In [8]:
MAX_IN_T5 = 256
MAX_TGT_T5 = 256

def format_for_t5(example):
    system, user, assistant = extract_dialog(example)
    if not user or not assistant:
        return {"input_ids": [], "labels": [], "valid": False}

    input_text = f"{system}\n\nВопрос: {user}" if system else f"Вопрос: {user}"
    input_ids = t5_tokenizer(input_text, truncation=True, max_length=MAX_IN_T5,
                              add_special_tokens=True)["input_ids"]
    target_ids = t5_tokenizer(assistant, truncation=True, max_length=MAX_TGT_T5,
                               add_special_tokens=True)["input_ids"]

    if len(target_ids) < 3:
        return {"input_ids": [], "labels": [], "valid": False}

    return {"input_ids": input_ids, "labels": target_ids, "valid": True}

Применение форматирования к train и val

In [9]:
# Для GPT
gpt_train = train_raw.map(format_for_gpt, remove_columns=train_raw.column_names)
gpt_val   = val_raw.map(format_for_gpt,   remove_columns=val_raw.column_names)

gpt_train = gpt_train.filter(lambda x: x["valid"]).remove_columns(["valid"])
gpt_val   = gpt_val.filter(lambda x: x["valid"]).remove_columns(["valid"])

# Для T5
t5_train = train_raw.map(format_for_t5, remove_columns=train_raw.column_names)
t5_val   = val_raw.map(format_for_t5,   remove_columns=val_raw.column_names)

t5_train = t5_train.filter(lambda x: x["valid"]).remove_columns(["valid"])
t5_val   = t5_val.filter(lambda x: x["valid"]).remove_columns(["valid"])

print("\nПосле фильтрации")
print(f"GPT train: {len(gpt_train)}, val: {len(gpt_val)}")
print(f"T5  train: {len(t5_train)}, val: {len(t5_val)}")

Filter: 100%|██████████| 1000/1000 [00:00<00:00, 12637.33 examples/s]


После фильтрации
GPT train: 18673, val: 926
T5  train: 19780, val: 983


### 3. Провека работы модели в zero-shot режиме.

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

gpt_model = gpt_model.to(device).eval()
t5_model = t5_model.to(device).eval()

Device: cuda


#### 3.1. zero-shot GPT

In [14]:
def gpt_generate(prompt_ids, max_new_tokens=100):
    inputs = torch.tensor([prompt_ids]).to(device)
    with torch.no_grad():
        outputs = gpt_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,             # greedy
            pad_token_id=gpt_tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][len(prompt_ids):]
    return gpt_tokenizer.decode(generated_ids, skip_special_tokens=True)

In [17]:
def gpt_generate_from_text(system, user, max_new_tokens=100):
    prompt = f"{system}\n\nПользователь: {user}\nАссистент:"
    prompt_ids = gpt_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    generated = gpt_generate(prompt_ids, max_new_tokens=max_new_tokens)
    return prompt, generated

examples = [
    ("Ты — полезный ассистент.", "Сколько будет 2 + 2?"),
    ("Ты — полезный ассистент.", "Объясни, что такое фотосинтез, простыми словами."),
    ("", "Придумай имя для кота."),
    ("Ты — вежливый ассистент.", "Переведи на английский: «Привет, как дела?»"),
]

for i, (system, user) in enumerate(examples):
    prompt, generated = gpt_generate_from_text(system, user)
    print(f"\nПример {i}")
    print(f"PROMPT (конец): {prompt[-200:]}")
    print(f"GENERATED: {generated}")


Пример 0
PROMPT (конец): Ты — полезный ассистент.

Пользователь: Сколько будет 2 + 2?
Ассистент:
GENERATED:  Сколько будет 2 + 2?
Пользователь: Сколько будет 2 + 2?
Ассистент: Сколько будет 2 + 2?
Пользователь: Сколько будет 2 + 2?
Ассистент: Сколько будет 2 + 2?
Пользователь: Сколько будет 2 + 2?
Ассистент: Сколько будет 2 + 2?
Пользователь: Сколько будет 2 + 2?
Ассистент: Сколько будет 2 + 2?
П

Пример 1
PROMPT (конец): Ты — полезный ассистент.

Пользователь: Объясни, что такое фотосинтез, простыми словами.
Ассистент:
GENERATED:  А что такое фотосинтез?
Пользователь: А что такое фотосинтез?
Ассистент: А что такое фотосинтез?
Пользователь: А что такое фотосинтез?
Ассистент: А что такое фотосинтез?
Пользователь: А что такое фотосинтез?
Ассистент: А что такое фотосинтез?
Пользователь: А что такое фотосинте

Пример 2
PROMPT (конец): 

Пользователь: Придумай имя для кота.
Ассистент:
GENERATED:  Ассистент, Ассистент, Ассистент, Ассистент, Ассистент, Ассистент, Ассистент, Ассистент, Ассист

#### 3.2. zero-shot T5

In [15]:
def t5_generate(input_ids, max_new_tokens=100):
    inputs = torch.tensor([input_ids]).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [18]:
def t5_generate_from_text(system, user, max_new_tokens=100):
    input_text = f"{system}\n\nВопрос: {user}" if system else f"Вопрос: {user}"
    input_ids = t5_tokenizer(input_text, add_special_tokens=True)["input_ids"]
    generated = t5_generate(input_ids, max_new_tokens=max_new_tokens)
    return input_text, generated

for i, (system, user) in enumerate(examples):
    input_text, generated = t5_generate_from_text(system, user)
    print(f"\nПример {i}")
    print(f"INPUT (конец): {input_text[-200:]}")
    print(f"GENERATED: {generated}")


Пример 0
INPUT (конец): Ты — полезный ассистент.

Вопрос: Сколько будет 2 + 2?
GENERATED: :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::

Пример 1
INPUT (конец): Ты — полезный ассистент.

Вопрос: Объясни, что такое фотосинтез, простыми словами.
GENERATED: ::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::

Пример 2
INPUT (конец): Вопрос: Придумай имя для кота.
GENERATED: Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос Вопрос

Пример 3
INPUT (конец): Ты — вежливый ассистент.

Вопрос: Переведи на английский: «Привет, как дела?»
GENERATED: ::,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


#### 3.2. Интерпретация результата

- `GPT` обучена продолжать текст, а не следовать инструкции. Она увидела запрос и решила, что это часть текста, который надо продолжить. На данном этапе она не выучила еще, как нужно дать ответ.
- `ruT5-base` обучена на дениойзинге (восстановлении текста), а не на диалогах. Поэтому мусор в результате ожидаем.

| Модель | Zero-shot результат       | Итог                              |
|--------|---------------------------|---------------------------------------------|
| GPT    | Повторяет запрос    | Не понимает формат, нужно дообучение        |
| T5     | Мусор из знаков           | Не понимает задачу, нужно дообучение        |

### 4. Метрики bleu и meteor исходной модели на валидационном сабсете.

In [19]:
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [ ]:
def compute_bleu(predictions, references):
    bleu = BLEU()
    score = bleu.corpus_score(predictions, [references])
    return score.score

def compute_meteor(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        if not pred.strip() or not ref.strip():
            continue
        scores.append(meteor_score([ref.split()], pred.split()))
    return sum(scores) / len(scores) if scores else 0.0

In [24]:
N_TEST_GPT = len(gpt_val)
gpt_preds = []
gpt_refs = []

for i in tqdm(range(N_TEST_GPT)):
    ex = gpt_val[i]
    prompt_ids = ex["input_ids"][:ex["prompt_len"]]
    
    generated = gpt_generate(prompt_ids, max_new_tokens=100)
    reference = gpt_tokenizer.decode(
        ex["input_ids"][ex["prompt_len"]:],
        skip_special_tokens=True
    ).strip()
    
    gpt_preds.append(generated.strip())
    gpt_refs.append(reference)

print("BLEU GPT (zero-shot):", compute_bleu(gpt_preds, gpt_refs))
print("METEOR GPT (zero-shot):", compute_meteor(gpt_preds, gpt_refs))

100%|██████████| 926/926 [11:56<00:00,  1.29it/s]


BLEU GPT (zero-shot): 3.5449047218397394
METEOR GPT (zero-shot): 0.06936083945809672


In [25]:
N_TEST_T5  = len(t5_val)

t5_preds = []
t5_refs = []

for i in tqdm(range(N_TEST_T5)):
    ex = t5_val[i]
    input_ids = ex["input_ids"]
    
    generated = t5_generate(input_ids, max_new_tokens=100)
    reference = t5_tokenizer.decode(
        [t for t in ex["labels"] if t != t5_tokenizer.pad_token_id],
        skip_special_tokens=True
    ).strip()
    
    t5_preds.append(generated.strip())
    t5_refs.append(reference)

print("BLEU T5 (zero-shot):", compute_bleu(t5_preds, t5_refs))
print("METEOR T5 (zero-shot):", compute_meteor(t5_preds, t5_refs))

100%|██████████| 983/983 [13:59<00:00,  1.17it/s]


BLEU T5 (zero-shot): 0.0848503211394825
METEOR T5 (zero-shot): 0.002704779541153249


In [28]:
with open("zero_shot_gpt.json", "w", encoding="utf-8") as f:
    json.dump({
        "preds": gpt_preds,
        "refs": gpt_refs,
        "bleu": compute_bleu(gpt_preds, gpt_refs),
        "meteor": compute_meteor(gpt_preds, gpt_refs),
    }, f, ensure_ascii=False, indent=2)

print("GPT сохранён")

GPT сохранён


In [29]:
with open("zero_shot_t5.json", "w", encoding="utf-8") as f:
    json.dump({
        "preds": t5_preds,
        "refs": t5_refs,
        "bleu": compute_bleu(t5_preds, t5_refs),
        "meteor": compute_meteor(t5_preds, t5_refs),
    }, f, ensure_ascii=False, indent=2)

print("T5 сохранён")

T5 сохранён


In [ ]:
with open("zero_shot_gpt.json", "r", encoding="utf-8") as f:
    gpt_data = json.load(f)

with open("zero_shot_t5.json", "r", encoding="utf-8") as f:
    t5_data = json.load(f)

print(f"GPT zero-shot: BLEU = {gpt_data['bleu']:.4f}, METEOR = {gpt_data['meteor']:.4f}")
print(f"T5  zero-shot: BLEU = {t5_data['bleu']:.4f}, METEOR = {t5_data['meteor']:.4f}")

#### Модели

- **Decoder-only:** `ai-forever/rugpt3small_based_on_gpt2` (125M)
- **Encoder-decoder:** `ai-forever/ruT5-base` (222M)

**Генерация:** `max_new_tokens=100`

#### Метрики zero-shot

| Модель        | BLEU | METEOR |
|---------------|------|--------|
| rugpt3small   | 3.54 | 0.069  |
| ruT5-base     | 0.08 | 0.003  |

### 5. Дообучение модели на обучающем сабсете.

### 5.1. Дообучение модели `rugpt3small_based_on_gpt2`

DataCollator для GPT

In [ ]:
@dataclass
class GPTDataCollator:
    tokenizer: Any
    max_length: int = 512

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = min(max(len(f["input_ids"]) for f in features), self.max_length)

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            ids = f["input_ids"][:max_len]
            prompt_len = min(f["prompt_len"], len(ids))

            # Padding справа
            pad_len = max_len - len(ids)
            padded = ids + [self.tokenizer.pad_token_id] * pad_len
            mask = [1] * len(ids) + [0] * pad_len

            label = [-100] * prompt_len + ids[prompt_len:] + [-100] * pad_len

            input_ids.append(padded)
            attention_mask.append(mask)
            labels.append(label)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

Training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt_finetuned",
    overwrite_output_dir=True,

    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,

    fp16=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,

    remove_unused_columns=False,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [37]:
gpt_collator = GPTDataCollator(tokenizer=gpt_tokenizer, max_length=512)

gpt_trainer = Trainer(
    model=gpt_model,
    args=training_args,
    train_dataset=gpt_train,
    eval_dataset=gpt_val,
    data_collator=gpt_collator,
)

In [39]:
gpt_trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss
500,2.169100,2.073405
1000,2.117800,2.005701
1500,1.883600,1.962632
2000,1.877700,1.936272


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tok

TrainOutput(global_step=2334, training_loss=2.0552610202844876, metrics={'train_runtime': 2609.5267, 'train_samples_per_second': 14.311, 'train_steps_per_second': 0.894, 'total_flos': 6566805127296000.0, 'train_loss': 2.0552610202844876, 'epoch': 1.9997857984363285})

In [ ]:
# Сохранение модели и токенизатора
gpt_save_dir = "./gpt_finetuned_final"
os.makedirs(gpt_save_dir, exist_ok=True)
gpt_trainer.save_model(gpt_save_dir)
gpt_tokenizer.save_pretrained(gpt_save_dir)

# Сохранение истории обучения
history = gpt_trainer.state.log_history
with open(f"{gpt_save_dir}/train_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"GPT сохранён в {gpt_save_dir}")

GPT сохранён в ./gpt_finetuned_final


### 5.2. Дообучение модели `ruT5-base`

DataCollator для T5

In [ ]:
@dataclass
class T5DataCollator:
    tokenizer: Any

    def __call__(self, features):
        # Паддинг для входа
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        batch = self.tokenizer.pad(
            {"input_ids": input_ids},
            padding=True,
            return_tensors="pt",
        )

        # Паддинг для labels
        max_label_len = max(len(l) for l in labels)
        pad_id = self.tokenizer.pad_token_id

        padded_labels = [
            l + [pad_id] * (max_label_len - len(l)) for l in labels
        ]
        padded_labels = [
            [t if t != pad_id else -100 for t in l] for l in padded_labels
        ]

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

TrainingArguments для T5

In [ ]:
t5_training_args = TrainingArguments(
    output_dir="./t5_finetuned",
    overwrite_output_dir=True,

    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=3e-4,
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,

    fp16=False,
    dataloader_num_workers=4,
    report_to="none",
    seed=42,

    remove_unused_columns=False,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [44]:
t5_collator = T5DataCollator(tokenizer=t5_tokenizer)

t5_trainer = Trainer(
    model=t5_model,
    args=t5_training_args,
    train_dataset=t5_train,
    eval_dataset=t5_val,
    data_collator=t5_collator,
)

In [45]:
t5_trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss,Validation Loss
500,3.069200,2.690440
1000,2.855400,2.542497
1500,2.533100,2.405111
2000,2.420500,2.328642


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
You're using a T5TokenizerFa

TrainOutput(global_step=2472, training_loss=2.8192823311271793, metrics={'train_runtime': 2516.8442, 'train_samples_per_second': 15.718, 'train_steps_per_second': 0.982, 'total_flos': 9742879045509120.0, 'train_loss': 2.8192823311271793, 'epoch': 1.9995955510616783})

In [46]:
t5_save_dir = "./t5_finetuned_final"
os.makedirs(t5_save_dir, exist_ok=True)
t5_trainer.save_model(t5_save_dir)
t5_tokenizer.save_pretrained(t5_save_dir)

with open(f"{t5_save_dir}/train_history.json", "w", encoding="utf-8") as f:
    json.dump(t5_trainer.state.log_history, f, ensure_ascii=False, indent=2)

print(f"T5 сохранён в {t5_save_dir}")

T5 сохранён в ./t5_finetuned_final


### 6-7. Проверка работу модели в сценарии близком к датасету и подсчет метрик `bleu` и `meteor` на валидационном сабсете.

### 6-7.1 GPT

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

gpt_ft_tokenizer = AutoTokenizer.from_pretrained("./gpt_finetuned_final")
gpt_ft_model = AutoModelForCausalLM.from_pretrained("./gpt_finetuned_final").to(device).eval()

if gpt_ft_tokenizer.pad_token is None:
    gpt_ft_tokenizer.pad_token = gpt_ft_tokenizer.eos_token

print("GPT fine-tuned загружена")
print(f"Параметров: {sum(p.numel() for p in gpt_ft_model.parameters())}")

GPT fine-tuned загружена
Параметров: 125231616


In [22]:
def gpt_ft_generate(prompt_ids, max_new_tokens=100):
    inputs = torch.tensor([prompt_ids]).to(device)
    attention_mask = torch.ones_like(inputs)
    with torch.no_grad():
        outputs = gpt_ft_model.generate(
            inputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=gpt_ft_tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][len(prompt_ids):]
    return gpt_ft_tokenizer.decode(generated_ids, skip_special_tokens=True)

In [23]:
def gpt_ft_generate_from_text(system, user, max_new_tokens=100):
    prompt = f"{system}\n\nПользователь: {user}\nАссистент:" if system else f"\nПользователь: {user}\nАссистент:"
    prompt_ids = gpt_ft_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    generated = gpt_ft_generate(prompt_ids, max_new_tokens=max_new_tokens)
    return prompt, generated

examples = [
    ("Ты — полезный ассистент.", "Сколько будет 2 + 2?"),
    ("Ты — полезный ассистент.", "Объясни, что такое фотосинтез, простыми словами."),
    ("", "Придумай имя для кота."),
    ("Ты — вежливый ассистент.", "Переведи на английский: «Привет, как дела?»"),
]

print("GPT fine-tuned на своих примерах")
for i, (system, user) in enumerate(examples):
    prompt, generated = gpt_ft_generate_from_text(system, user)
    print(f"\nПример {i}")
    print(f"USER: {user}")
    print(f"GENERATED: {generated}")

GPT fine-tuned на своих примерах

Пример 0
USER: Сколько будет 2 + 2?
GENERATED: Чтобы найти количество 2 + 2, нам нужно найти количество 2 + 2 = 2.

Чтобы найти количество 2 + 2, нам нужно найти количество 2 + 2 = 2.

Чтобы найти количество 2 + 2 = 2, нам нужно найти количество 2 + 2 = 2.

Чтобы найти количество 2 + 2 = 2, нам нужно найти количество 2 + 2 = 2.

Чтобы найти количество 2 + 2 = 2, нам нужно найти количество 2 + 2 = 2.


Пример 1
USER: Объясни, что такое фотосинтез, простыми словами.
GENERATED: Фотосинтез — это процесс, который включает в себя преобразование органических веществ в химические вещества, которые затем используются для производства энергии. Это включает в себя использование энергии солнца, ветра и других источников энергии, таких как солнечная и ветровая энергия.

Пример 2
USER: Придумай имя для кота.
GENERATED: Кот — это кот, который любит играть с другими животными, особенно с кошками.

Пример 3
USER: Переведи на английский: «Привет, как дела?»
GENERATED: П

In [ ]:
gpt_ft_preds = []
gpt_ft_refs = []

for i in tqdm(range(len(gpt_val))):
    ex = gpt_val[i]
    prompt_ids = ex["input_ids"][:ex["prompt_len"]]
    
    generated = gpt_ft_generate(prompt_ids, max_new_tokens=100)
    reference = gpt_ft_tokenizer.decode(
        ex["input_ids"][ex["prompt_len"]:],
        skip_special_tokens=True
    ).strip()
    
    gpt_ft_preds.append(generated.strip())
    gpt_ft_refs.append(reference)

bleu_gpt_ft = compute_bleu(gpt_ft_preds, gpt_ft_refs)
meteor_gpt_ft = compute_meteor(gpt_ft_preds, gpt_ft_refs)

print(f"\nGPT fine-tuned: BLEU = {bleu_gpt_ft:.4f}, METEOR = {meteor_gpt_ft:.4f}")

with open("finetuned_gpt.json", "w", encoding="utf-8") as f:
    json.dump({
        "preds": gpt_ft_preds,
        "refs": gpt_ft_refs,
        "bleu": bleu_gpt_ft,
        "meteor": meteor_gpt_ft,
    }, f, ensure_ascii=False, indent=2)

print("Сохранено в finetuned_gpt.json")

100%|██████████| 926/926 [05:59<00:00,  2.58it/s]



GPT fine-tuned: BLEU = 3.9729, METEOR = 0.1438
Сохранено в finetuned_gpt.json


### 6-7.2 T5

In [24]:
t5_ft_tokenizer = AutoTokenizer.from_pretrained("./t5_finetuned_final")
t5_ft_model = AutoModelForSeq2SeqLM.from_pretrained("./t5_finetuned_final").to(device).eval()

print("T5 fine-tuned загружена")
print(f"Параметров: {sum(p.numel() for p in t5_ft_model.parameters())}")

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


T5 fine-tuned загружена
Параметров: 222903552


In [25]:
def t5_ft_generate(input_ids, max_new_tokens=100):
    inputs = torch.tensor([input_ids]).to(device)
    attention_mask = torch.ones_like(inputs)
    with torch.no_grad():
        outputs = t5_ft_model.generate(
            inputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return t5_ft_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [28]:
def t5_ft_generate_from_text(system, user, max_new_tokens=100):
    input_text = f"{system}\n\nВопрос: {user}" if system else f"Вопрос: {user}"
    input_ids = t5_ft_tokenizer(input_text, add_special_tokens=True)["input_ids"]
    generated = t5_ft_generate(input_ids, max_new_tokens=max_new_tokens)
    return input_text, generated

for i, (system, user) in enumerate(examples):
    _, ft = t5_ft_generate_from_text(system, user)
    print(f"\nПример {i}")
    print(f"USER: {user}")
    print(f"FINE-TUNED: {ft}")


Пример 0
USER: Сколько будет 2 + 2?
FINE-TUNED: Чтобы найти количество + 2, нам нужно найти количество + 2, которое мы хотим получить. Мы можем использовать формулу: 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 = 2 + 2 =

Пример 1
USER: Объясни, что такое фотосинтез, простыми словами.
FINE-TUNED: Фотосинтез — это процесс, при котором молекулы света преобразуют свет в энергию, необходимую для производства энергии. Фотосинтез — это процесс, при котором молекулы света преобразуют свет в энергию, которую они преобразуют в энергию, которую они преобразуют в энергию, которую они получают из энергии, которую они получают из энергии, которую они получают из энергии, которую они получают из энергии, которую они получают из энергии, которую они получают из энергии, которую они получают из энергии, которую

Пример 2
USER: Придумай имя для кота.
FINE-TUNED: «Кот»

Пример 3
USER: Переведи на английский

In [55]:
t5_ft_preds = []
t5_ft_refs = []

for i in tqdm(range(len(t5_val))):
    ex = t5_val[i]
    input_ids = ex["input_ids"]
    
    generated = t5_ft_generate(input_ids, max_new_tokens=100)
    reference = t5_ft_tokenizer.decode(
        [t for t in ex["labels"] if t != t5_ft_tokenizer.pad_token_id],
        skip_special_tokens=True
    ).strip()
    
    t5_ft_preds.append(generated.strip())
    t5_ft_refs.append(reference)

bleu_t5_ft = compute_bleu(t5_ft_preds, t5_ft_refs)
meteor_t5_ft = compute_meteor(t5_ft_preds, t5_ft_refs)

print(f"\nT5 fine-tuned: BLEU = {bleu_t5_ft:.4f}, METEOR = {meteor_t5_ft:.4f}")

with open("finetuned_t5.json", "w", encoding="utf-8") as f:
    json.dump({
        "preds": t5_ft_preds,
        "refs": t5_ft_refs,
        "bleu": bleu_t5_ft,
        "meteor": meteor_t5_ft,
    }, f, ensure_ascii=False, indent=2)

print("Сохранено в finetuned_t5.json")

100%|██████████| 983/983 [15:12<00:00,  1.08it/s]



T5 fine-tuned: BLEU = 5.9107, METEOR = 0.1305
Сохранено в finetuned_t5.json


### 8. Сравнение метрик обученной и исходной моделей.

In [ ]:
with open("zero_shot_gpt.json", "r", encoding="utf-8") as f:
    zs_gpt = json.load(f)
with open("zero_shot_t5.json", "r", encoding="utf-8") as f:
    zs_t5 = json.load(f)
with open("finetuned_gpt.json", "r", encoding="utf-8") as f:
    ft_gpt = json.load(f)
with open("finetuned_t5.json", "r", encoding="utf-8") as f:
    ft_t5 = json.load(f)

print(f"{'Модель':<20} {'BLEU (zero)':<15} {'BLEU (ft)':<15} {'METEOR (zero)':<15} {'METEOR (ft)':<15}")
print(f"{'rugpt3small':<20} {zs_gpt['bleu']:<15.4f} {ft_gpt['bleu']:<15.4f} {zs_gpt['meteor']:<15.4f} {ft_gpt['meteor']:<15.4f}")
print(f"{'ruT5-base':<20} {zs_t5['bleu']:<15.4f} {ft_t5['bleu']:<15.4f} {zs_t5['meteor']:<15.4f} {ft_t5['meteor']:<15.4f}")


Модель               BLEU (zero)     BLEU (ft)       METEOR (zero)   METEOR (ft)    
rugpt3small          3.5449          3.9729          0.0694          0.1438         
ruT5-base            0.0849          5.9107          0.0027          0.1305         


### Итог

#### Результаты
- BLEU в %
- METEOR в долях

| Модель | BLEU (zero-shot) | BLEU (fine-tuned) | METEOR (zero-shot) | METEOR (fine-tuned) |
|---|---|---|---|---|
| rugpt3small (decoder-only) | 3.545 | 3.973 | 0.069 | 0.144 |
| ruT5-base (encoder-decoder) | 0.085 | 5.911 | 0.003 | 0.131 |

**Обе модели улучшились после дообучения.**.

- **ruT5-base**: BLEU вырос в 70 раз (с 0.08 до 5.91), METEOR увеличился в 48 раз (с 0.003 до 0.131). Encoder-decoder архитектура эффективно выучила формат «вход - ответ». Более уверенно генерируется связный верный ответ.
- **rugpt3small** увеличила METEOR с 0.069 до 0.144 (в 2 раза), но BLEU вырос слабо (с 3.54 до 3.97). Decoder-only модель чаще генерирует связный, но не отвечающий запросу текст.
- **По BLEU лидирует T5**, по METEOR - GPT, но разница по METEOR минимальна (0.013).

Абсолютные значения BLEU низкие у обеих моделей. Это объясняется наличием всего двух эпох для моделей 125M/222M на разнородном датасете.  

#### Вывод

Обе архитектуры удалось улучшить и подстроить под диалоговую задачу за счет finetuning, метрики улучшились. **Encoder-decoder (T5) показала лучшее качество на задаче генерации ответа**, что ожидаемо, поскольку такие модели лучше подходят для преобразования текста «запрос - ответ», чем decoder-only модели. Особенно это заметно при малых объёмах данных и при ограниченных вычислительных ресурсах.